# 🌐 Threat Shield — Flask API for Real-Time URL Classification

**Purpose:** This notebook documents the Flask-based REST API that serves the trained Random Forest model for real-time malicious URL detection.  
**Integration:** Works with the Chrome Extension and a standalone web interface.  
**Architecture:**

```
User (Browser / Extension) → HTTP POST /predict → Flask API
    → Feature Extraction → Random Forest Model → Classification Result
    → JSON Response {result, predicted_class, confidence}
```

---

> ⚠️ **Note:** This notebook is for **documentation and demonstration purposes**. The Flask server is designed to run as `app.py` from the terminal. Code cells below walk through each component.

## 1 · Import Libraries

In [ ]:
from flask import Flask, request, jsonify, render_template
import pandas as pd
import re
from urllib.parse import urlparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from flask_cors import CORS

print("✅ All libraries imported.")

## 2 · Flask App Initialization

The Flask app is created with **CORS enabled** so the Chrome Extension (running on a different origin) can make cross-origin requests to the API.

In [ ]:
app = Flask(__name__)
CORS(app)  # Enable Cross-Origin Resource Sharing

print("✅ Flask app initialized with CORS.")

## 3 · Data Loading & Sampling

The dataset is loaded at server startup. To manage memory, we apply **stratified sampling** — capping each class at 100,000 rows while preserving class proportions.

In [ ]:
# Load the dataset
df = pd.read_csv('malicious_phish.csv')
print(f"Loaded: {df.shape[0]:,} rows")

# Stratified sampling to manage memory
MAX_PER_CLASS = 100000
if df.groupby('type').size().max() > MAX_PER_CLASS:
    df = df.groupby('type', group_keys=False).apply(
        lambda x: x.sample(min(len(x), MAX_PER_CLASS), random_state=42)
    ).reset_index(drop=True)
    print(f"After stratified sampling: {df.shape[0]:,} rows")

print(f"\nClass distribution:")
print(df['type'].value_counts())

## 4 · Feature Engineering Functions

The same **20 features** used during training are extracted from each incoming URL at prediction time. This ensures consistency between training and inference.

| Feature | What it captures |
|:---|:---|
| `use_of_ip` | URLs using raw IP addresses instead of domain names |
| `abnormal_url` | Hostname appearing in unexpected positions |
| `count.`, `count@`, `count-` | Special character frequencies (obfuscation signals) |
| `short_url` | Use of URL shortening services (bit.ly, tinyurl, etc.) |
| `sus_url` | Presence of phishing keywords (login, bank, account, etc.) |
| `url_length`, `hostname_length` | Structural length metrics |

In [ ]:
# ── Feature Extraction Functions ──

def contains_ip_address(url):
    """Detect raw IP addresses in URL (e.g., http://192.168.1.1/login)."""
    return 1 if re.search(r'(\d{1,3}\.){3}\d{1,3}', url) else 0

def abnormal_url(url):
    """Check if hostname appears in an abnormal position."""
    hostname = urlparse(url).hostname
    return 1 if hostname and hostname in url else 0

def count_dot(url):     return url.count('.')
def count_www(url):     return url.count('www')
def count_atrate(url):  return url.count('@')
def no_of_dir(url):     return urlparse(url).path.count('/')
def no_of_embed(url):   return urlparse(url).path.count('//')
def count_https(url):   return url.count('https')
def count_http(url):    return url.count('http')
def count_per(url):     return url.count('%')
def count_ques(url):    return url.count('?')
def count_hyphen(url):  return url.count('-')
def count_equal(url):   return url.count('=')
def url_length(url):    return len(str(url))
def hostname_length(url): return len(urlparse(url).netloc)

def suspicious_words(url):
    """Detect phishing-related keywords in URL."""
    return 1 if re.search('login|bank|account|update|free|bonus', url) else 0

def digit_count(url):   return sum(c.isnumeric() for c in url)
def letter_count(url):  return sum(c.isalpha() for c in url)

def fd_length(url):
    """Length of the first directory in URL path."""
    try:
        return len(urlparse(url).path.split('/')[1])
    except:
        return 0

def shortening_service(url):
    """Detect known URL shortening services."""
    return 1 if re.search('bit\.ly|tinyurl|t\.co|goo\.gl', url) else 0

print("✅ Feature functions defined.")

## 5 · Apply Features & Train Model

At server startup, we extract all features from the training data, encode labels, split, and train the Random Forest model.

In [ ]:
# Apply all 20 features
df['use_of_ip']          = df['url'].apply(contains_ip_address)
df['abnormal_url']       = df['url'].apply(abnormal_url)
df['count.']             = df['url'].apply(count_dot)
df['count-www']          = df['url'].apply(count_www)
df['count@']             = df['url'].apply(count_atrate)
df['count_dir']          = df['url'].apply(no_of_dir)
df['count_embed_domian'] = df['url'].apply(no_of_embed)
df['short_url']          = df['url'].apply(shortening_service)
df['count-https']        = df['url'].apply(count_https)
df['count-http']         = df['url'].apply(count_http)
df['count%']             = df['url'].apply(count_per)
df['count?']             = df['url'].apply(count_ques)
df['count-']             = df['url'].apply(count_hyphen)
df['count=']             = df['url'].apply(count_equal)
df['url_length']         = df['url'].apply(url_length)
df['hostname_length']    = df['url'].apply(hostname_length)
df['sus_url']            = df['url'].apply(suspicious_words)
df['fd_length']          = df['url'].apply(fd_length)
df['count-digits']       = df['url'].apply(digit_count)
df['count-letters']      = df['url'].apply(letter_count)

print(f"✅ Features extracted. DataFrame shape: {df.shape}")

In [ ]:
# Encode labels
lb = LabelEncoder()
df['url_type'] = lb.fit_transform(df['type'])

# Define features and target
feature_cols = ['use_of_ip', 'abnormal_url', 'count.', 'count-www', 'count@',
                'count_dir', 'count_embed_domian', 'short_url', 'count-https',
                'count-http', 'count%', 'count?', 'count-', 'count=',
                'url_length', 'hostname_length', 'sus_url', 'fd_length',
                'count-digits', 'count-letters']

X = df[feature_cols]
y = df['url_type']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

from sklearn.metrics import accuracy_score
print(f"✅ Model trained. Test accuracy: {accuracy_score(y_test, rf.predict(X_test)):.4f}")

## 6 · Feature Extraction for Prediction

When a URL arrives via the API, we extract the same 20 features and feed them to the trained model.

In [ ]:
def extract_features(url):
    """Extract all 20 features from a single URL for prediction."""
    return [
        contains_ip_address(url),
        abnormal_url(url),
        count_dot(url),
        count_www(url),
        count_atrate(url),
        no_of_dir(url),
        no_of_embed(url),
        shortening_service(url),
        count_https(url),
        count_http(url),
        count_per(url),
        count_ques(url),
        count_hyphen(url),
        count_equal(url),
        url_length(url),
        hostname_length(url),
        suspicious_words(url),
        fd_length(url),
        digit_count(url),
        letter_count(url)
    ]

print("✅ extract_features() defined.")

## 7 · Trusted Domain Whitelist

To reduce false positives on well-known legitimate websites, we maintain a **whitelist of trusted domains**. If a URL matches, we skip the ML pipeline and return `SAFE` immediately.

In [ ]:
TRUSTED_DOMAINS = {
    'google.com', 'youtube.com', 'gmail.com', 'accounts.google.com',
    'amazon.com', 'aws.amazon.com',
    'facebook.com', 'instagram.com', 'whatsapp.com',
    'microsoft.com', 'live.com', 'outlook.com', 'office.com',
    'login.microsoftonline.com', 'microsoftonline.com',
    'apple.com', 'icloud.com',
    'twitter.com', 'x.com',
    'linkedin.com',
    'github.com',
    'netflix.com',
    'paypal.com',
    'wikipedia.org',
    'yahoo.com',
    'reddit.com',
}

def is_trusted_domain(url):
    """Check if the URL's hostname matches or is a subdomain of a trusted domain."""
    try:
        hostname = urlparse(url).hostname
        if not hostname:
            return False
        hostname = hostname.lower()
        for domain in TRUSTED_DOMAINS:
            if hostname == domain or hostname.endswith('.' + domain):
                return True
        return False
    except Exception:
        return False

# Test examples
test_urls = [
    'https://www.google.com/search?q=test',
    'https://mail.google.com/inbox',
    'http://suspicious-site.xyz/login'
]
for url in test_urls:
    print(f"  {url:50s} → Trusted: {is_trusted_domain(url)}")

## 8 · API Routes & Prediction Logic

The API exposes two endpoints:

| Route | Method | Purpose |
|:---|:---|:---|
| `/` | GET | Serves the web interface (`index.html`) |
| `/predict` | POST | Accepts a URL in JSON body, returns classification |

In [ ]:
# ── Route: Home Page ──
@app.route('/')
def home():
    """Serve the web interface."""
    return render_template('index.html')

# ── Route: Prediction API ──
@app.route('/predict', methods=['POST'])
def predict():
    """Classify a URL as safe or malicious.
    
    Request Body (JSON):
        {"url": "https://example.com"}
    
    Response (JSON):
        {
            "result": "URL IS SAFE!" or "URL IS MALICIOUS!",
            "result_str": same as result (compatibility),
            "predicted_class": "benign"/"phishing"/"malware"/"defacement",
            "confidence": 0-100
        }
    """
    data = request.json
    url = data.get('url')

    # Step 1: Check trusted domain whitelist
    if is_trusted_domain(url):
        msg = 'URL IS SAFE!'
        return jsonify({
            'result': msg,
            'result_str': msg,
            'predicted_class': 'benign',
            'confidence': 100
        })

    # Step 2: Extract features and classify
    f = extract_features(url)
    predicted_index = rf.predict([f])[0]
    predicted_class = lb.inverse_transform([predicted_index])[0]

    # Step 3: Compute confidence score
    probabilities = rf.predict_proba([f])[0]
    confidence = round(float(max(probabilities)) * 100)

    # Step 4: Determine verdict
    if predicted_class in ['benign', 'defacement']:
        msg = 'URL IS SAFE!'
    else:
        msg = 'URL IS MALICIOUS!'

    return jsonify({
        'result': msg,
        'result_str': msg,
        'predicted_class': predicted_class,
        'confidence': confidence
    })

print("✅ Routes defined: / (GET) and /predict (POST)")

## 9 · Demo — Simulated Predictions

Let's test the prediction pipeline without starting the server:

In [ ]:
demo_urls = [
    'https://www.google.com/search?q=python',
    'http://192.168.1.1/admin/login.php',
    'https://www.wikipedia.org/wiki/Machine_learning',
    'http://free-prize-winner.xyz/claim?id=12345&token=abc',
    'http://bit.ly/3xF2kL9',
    'https://login-bankofamerica-secure.com/verify-account'
]

print(f"{'URL':65s} {'Prediction':12s} {'Confidence':10s} {'Verdict'}")
print('─' * 110)

for url in demo_urls:
    # Check trusted domains first
    if is_trusted_domain(url):
        print(f"{url:65s} {'benign':12s} {'100%':10s} ✅ SAFE (trusted)")
        continue

    f = extract_features(url)
    predicted_index = rf.predict([f])[0]
    predicted_class = lb.inverse_transform([predicted_index])[0]
    probabilities = rf.predict_proba([f])[0]
    confidence = round(float(max(probabilities)) * 100)

    verdict = '✅ SAFE' if predicted_class in ['benign', 'defacement'] else '🚨 MALICIOUS'
    print(f"{url:65s} {predicted_class:12s} {confidence}%{'':<8s} {verdict}")

## 10 · Running the Server

To start the Flask API server, run the following command in your terminal:

```bash
python app.py
```

The server starts at `http://127.0.0.1:5000` and exposes:
- **Web UI** at `/`
- **Prediction API** at `/predict` (POST)

### Example cURL Request
```bash
curl -X POST http://127.0.0.1:5000/predict \
  -H "Content-Type: application/json" \
  -d '{"url": "http://suspicious-login-page.xyz/verify"}'
```

### Expected Response
```json
{
    "result": "URL IS MALICIOUS!",
    "result_str": "URL IS MALICIOUS!",
    "predicted_class": "phishing",
    "confidence": 87
}
```

---

## 📌 Conclusion

| Component | Description |
|:---|:---|
| **Framework** | Flask with CORS for cross-origin Chrome Extension support |
| **Model** | Random Forest (100 trees), trained on-startup |
| **Features** | 20 handcrafted URL features extracted in real-time |
| **Safeguard** | Trusted domain whitelist to reduce false positives |
| **Output** | JSON with verdict, predicted class, and confidence score |

### System Architecture
```
┌──────────────┐     ┌─────────────┐     ┌──────────────────┐
│   Chrome     │────▶│  Flask API  │────▶│  Random Forest   │
│  Extension   │◀────│  /predict   │◀────│  Classifier      │
└──────────────┘     └─────────────┘     └──────────────────┘
       │                    │                      │
   User sees           Trusted Domain        20 URL Features
   result alert        Whitelist Check       Extracted & Fed
```

---

## 🔮 Future Work

- 🚀 **Production Deployment**: Migrate to Gunicorn/WSGI for scalable serving
- 📦 **Pre-trained Model Loading**: Use `pickle.load()` instead of retraining on each startup
- 🔄 **Model Updates**: Periodic retraining with fresh threat data
- 📊 **Analytics Dashboard**: Track prediction history and detection patterns
- 🔐 **Rate Limiting & Authentication**: Protect the API from abuse

---
*Threat Shield — IGDTUW IT Workshop Project*